# Constrained Macro Scenario Optimisation

**Goal**: Find the *most plausible* deviation from the historical macro environment  
that causes the portfolio loss to exceed **10,000 EUR million**.

Formally:

$$\min_{\boldsymbol{\Delta} \in \mathbb{R}^5} \; D_M^2 = \boldsymbol{\Delta}^\top \boldsymbol{\Sigma}^{-1} \boldsymbol{\Delta}$$

$$\text{subject to} \quad L(\boldsymbol{\Delta}) \geq 10{,}000 \text{ EUR million}$$

where $\boldsymbol{\Delta} = \mathbf{x} - \boldsymbol{\mu}$ is the deviation of the macro scenario $\mathbf{x}$ from its  
historical mean $\boldsymbol{\mu}$, and $\boldsymbol{\Sigma}$ is the historical covariance matrix.

---

### Methodology

**Step 1 — Macro → stressed PD** (long-run sensitivity model):

From the OLS sensitivity analysis (current + 4 lagged macro factors), we compute the  
**cumulative impulse-response** beta for each variable by summing current and all lag coefficients.  
This represents the steady-state PD change after a *permanent* macro shift:

$$b_{s,j}^{\text{total}} = \sum_{k=0}^{4} \hat{\beta}_{s,j}^{(k)}$$

The stressed PD of exposure $i$ (in sector $s$) is then:

$$\text{PD}_i^*(\boldsymbol{\Delta}) = \sigma\!\left(\text{logit}(\text{PD}_i^0) + \mathbf{b}_{s(i)}^{\text{total}} \cdot \boldsymbol{\Delta}\right)$$

**Step 2 — Portfolio loss** (Gordy 2003 analytical approximation at 99.9%):

$$L(\boldsymbol{\Delta}) = \sum_{i=1}^{n} \text{EAD}_i \cdot \text{LGD} \cdot \Phi\!\left(\frac{\Phi^{-1}(\text{PD}_i^*) + \sqrt{\rho_i}\,\Phi^{-1}(q)}{\sqrt{1-\rho_i}}\right)$$

**Solver**: SLSQP (Sequential Least Squares Programming), scipy.optimize.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import norm, chi2
from scipy.optimize import minimize
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from pd_pipeline.basel import asset_correlation_formula
from pd_pipeline import data, config

# ── Problem parameters ─────────────────────────────────────────────────────────
LGD             = 0.40
LOSS_QUANTILE   = 0.999       # Gordy tail quantile
LOSS_THRESHOLD  = 7_000.0    # EUR million
N_LAGS          = 4           # number of quarterly lags in the OLS

MACRO_VARS = ['GDP_Growth', 'Interest_Rate', 'Brent_Oil', 'Fuel_Index', 'CPI']
GPR_VARS   = ['GPR_Global']
ALL_VARS   = MACRO_VARS + GPR_VARS
N_VARS     = len(ALL_VARS)

VAR_LABELS = {
    'GDP_Growth'    : 'GDP',
    'Interest_Rate' : 'IR',
    'Brent_Oil'     : 'Oil',
    'Fuel_Index'    : 'CIX',
    'CPI'           : 'CPI Index',
    'GPR_Global'    : 'GPR',
}

print(f'LGD            = {LGD:.0%}')
print(f'Gordy quantile = {LOSS_QUANTILE:.1%}')
print(f'Loss threshold = {LOSS_THRESHOLD:,.0f} EUR million')

## 1. Portfolio Exposures

In [ ]:
df_port = pd.read_csv('../data/portfolio_simulation.csv')
df_port['rho']        = df_port['pd'].apply(asset_correlation_formula)
df_port['pd_clipped'] = np.clip(df_port['pd'], 1e-9, 1 - 1e-9)

print(f'Exposures   : {len(df_port):,}')
print(f'Total EAD   : {df_port["ead_eur_m"].sum():,.1f} EUR million')
print(f'Average PD  : {df_port["pd"].mean()*100:.2f}%')
print()
print(df_port['sector'].value_counts().rename('count').to_frame().to_string())

## 2. Long-Run Sensitivity Coefficients

The OLS regressions include the current period plus 4 lagged values of each macro variable.  
For a **permanent** macro shock $\Delta_j$, every lag also shifts by $\Delta_j$, so the  
long-run (steady-state) logit-PD response is the **sum of all coefficients** across lags.

In [ ]:
df_sens = pd.read_csv('../01_pd_analysis/sensitivity_results_with_CI.csv')
df_s12  = df_sens[df_sens['PD_Horizon'] == '12_month'].copy()

# Sum current + lagged betas for each variable
sens_total = {}
for _, row in df_s12.iterrows():
    sector = row['Sector']
    betas = {}
    for v in MACRO_VARS:
        betas[v] = row[f'\u03b2_{v}'] + sum(
            row.get(f'\u03b2_{v}_lag{k}', 0) for k in range(1, N_LAGS + 1)
        )
    for v in GPR_VARS:
        betas[v] = row[f'\u03b4_{v}'] + sum(
            row.get(f'\u03b4_{v}_lag{k}', 0) for k in range(1, N_LAGS + 1)
        )
    sens_total[sector] = betas

# Print long-run sensitivity table
tbl = pd.DataFrame(sens_total).T[ALL_VARS].rename(columns=VAR_LABELS)
print('Long-run (summed) sensitivity coefficients  [Δ logit-PD per unit of Δ macro var]')
print(tbl.round(4).to_string())

In [ ]:
# Filter portfolio to sectors with sensitivities
df_valid = df_port[df_port['sector'].isin(sens_total)].copy().reset_index(drop=True)
print(f'Exposures matched to sensitivity sectors: {len(df_valid):,} / {len(df_port):,}')

n_exp = len(df_valid)
pd0   = df_valid['pd_clipped'].values
rho   = df_valid['rho'].values
ead   = df_valid['ead_eur_m'].values

# Build sensitivity matrix B_total : shape (n_exp, n_vars)
# Each row i contains the long-run beta vector for exposure i's sector
B_total = np.zeros((n_exp, N_VARS))
for i, (_, row) in enumerate(df_valid.iterrows()):
    s = row['sector']
    for j, v in enumerate(ALL_VARS):
        B_total[i, j] = sens_total[s][v]

# Pre-compute constants for the Gordy formula
sqrt_rho   = np.sqrt(rho)
sqrt_1mrho = np.sqrt(1 - rho)
inv_q      = norm.ppf(LOSS_QUANTILE)
logit_pd0  = np.log(pd0 / (1 - pd0))   # logit(PD_base)


## 3. Historical Distribution Parameters

In [ ]:
macro_frames = data.load_macro_data(
    gdp_path      = '../data/macro/global_gdp_monthly.csv',
    interest_path = '../data/macro/intrest FRED.csv',
    brent_path    = '../data/macro/brent_oil_monthly.csv',
    fuel_path     = '../data/macro/fuel_index_monthly.csv',
    cpi_path      = '../data/macro/global_cpi_monthly.csv',
    verbose       = False,
)
df_gpr    = data.load_gpr_data('../data/geopolitical/data_gpr_Data_GPR.csv', verbose=False)
df_merged = data.merge_macro_data(macro_frames, df_gpr)

cov_df, _, mean_series = data.summarize_macro_data(
    df_merged, config.ALL_PREDICTOR_COLS, verbose=False
)

mu        = mean_series.values          # (6,)
Sigma     = cov_df.values               # (6×6)
Sigma_inv = np.linalg.inv(Sigma)
stds      = np.sqrt(np.diag(Sigma))

print('Historical mean vector μ  (= scenario baseline):')
for v, m, s in zip(ALL_VARS, mu, stds):
    print(f'  {v:30s}: μ = {m:>10.3f},  σ = {s:>8.3f}')

## 4. Loss Function and Baseline

The loss function maps a **deviation vector** $\boldsymbol{\Delta}$ from the historical mean  
to the Gordy 99.9% portfolio loss.  At $\boldsymbol{\Delta} = \mathbf{0}$ the portfolio uses its  
base PDs (no macro stress).

In [ ]:
def portfolio_loss(delta: np.ndarray) -> float:
    """Gordy 99.9% portfolio loss (EUR million) under macro deviation Δ from μ."""
    adj        = np.clip(B_total @ delta, -50, 50)
    logit_pd_s = logit_pd0 + adj
    pd_s       = 1 / (1 + np.exp(-logit_pd_s))
    pd_s       = np.clip(pd_s, 1e-9, 1 - 1e-9)
    inv_pd_s   = norm.ppf(pd_s)
    cond_pd    = norm.cdf((inv_pd_s + sqrt_rho * inv_q) / sqrt_1mrho)
    return float(np.sum(ead * LGD * cond_pd))


def portfolio_loss_grad(delta: np.ndarray) -> np.ndarray:
    """Analytical gradient of portfolio_loss w.r.t. delta.

    Chain rule through stressed PDs:
        ∂L/∂Δ = B_total.T @ (EAD · LGD · φ(a) · PD*(1−PD*) / (√(1−ρ) · φ(Φ⁻¹(PD*))))
    """
    adj     = np.clip(B_total @ delta, -50, 50)
    pd_s    = np.clip(1 / (1 + np.exp(-(logit_pd0 + adj))), 1e-9, 1 - 1e-9)
    inv_pds = norm.ppf(pd_s)
    a       = (inv_pds + sqrt_rho * inv_q) / sqrt_1mrho
    phi_a   = norm.pdf(a)
    phi_inv = norm.pdf(inv_pds)
    var_pd  = pd_s * (1 - pd_s)
    return B_total.T @ (ead * LGD * phi_a * var_pd / (sqrt_1mrho * phi_inv))


def mahal_sq(delta: np.ndarray) -> float:
    """Squared Mahalanobis distance of Δ."""
    return float(delta @ Sigma_inv @ delta)


def mahal_sq_grad(delta: np.ndarray) -> np.ndarray:
    """Analytical gradient of the squared Mahalanobis distance."""
    return 2 * Sigma_inv @ delta


# Baseline (no stress)
loss_base = portfolio_loss(np.zeros(N_VARS))
print(f'Baseline loss (Δ = 0, q = 99.9%): {loss_base:>8,.0f} EUR million')
print(f'Loss threshold                   : {LOSS_THRESHOLD:>8,.0f} EUR million')
print(f'Required stress increase         : {LOSS_THRESHOLD - loss_base:>8,.0f} EUR million  '
      f'(+{(LOSS_THRESHOLD/loss_base - 1)*100:.0f}%)')

# Gradient check
_d_test = np.array([-1.0, +0.5, +1.5, -0.5, +0.8, +1.2]) * stds
_g_anal = portfolio_loss_grad(_d_test)
_eps    = 1e-5
_g_fd   = np.array([(portfolio_loss(_d_test + _eps*np.eye(N_VARS)[j])
                      - portfolio_loss(_d_test - _eps*np.eye(N_VARS)[j])) / (2*_eps)
                     for j in range(N_VARS)])
_rel_err = np.abs(_g_anal - _g_fd) / (np.abs(_g_fd) + 1e-12)
print(f'\nConstraint gradient check (analytical vs FD):')
for v, ga, gf, re in zip(ALL_VARS, _g_anal, _g_fd, _rel_err):
    print(f'  {v:30s}  anal={ga:+.4f}  FD={gf:+.4f}  rel_err={re:.2e}')

## 5. Loss Surface — Single-Factor Sensitivity

Each panel shows how the portfolio loss responds to a ±3σ shift in one macro  
variable while all others remain at their historical mean.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()

for idx, var in enumerate(ALL_VARS):
    ax  = axes_flat[idx]
    j   = ALL_VARS.index(var)
    std = stds[j]

    sweep  = np.linspace(-3 * std, 3 * std, 80)
    losses = []
    for dev in sweep:
        delta_test      = np.zeros(N_VARS)
        delta_test[j]   = dev
        losses.append(portfolio_loss(delta_test))

    ax.plot(sweep / std, losses, lw=2, color='steelblue')
    ax.axhline(LOSS_THRESHOLD, color='crimson', ls='--', lw=1.5,
               label=f'{LOSS_THRESHOLD/1000:.0f}k M threshold')
    ax.axhline(loss_base, color='grey', ls=':', lw=1.2, label='Baseline')
    ax.axvline(0, color='black', lw=0.8, ls=':')
    ax.set_xlabel(f'Δ{var}  (σ units)', fontsize=9)
    ax.set_ylabel('Loss (EUR m)', fontsize=9)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v/1000:.0f}k'))
    ax.set_title(VAR_LABELS[var], fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle('Portfolio Loss vs. Single-Factor Perturbations (others held at μ)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('constrained_opt_loss_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → constrained_opt_loss_surface.png')

## 6. Constrained Optimisation

We minimise the **squared Mahalanobis distance** of $\boldsymbol{\Delta}$ from zero  
(equivalently, the deviation from the historical mean) subject to the loss constraint.

Three starting points are used for robustness.

In [ ]:
constraint = {
    'type': 'ineq',
    'fun' : lambda delta: portfolio_loss(delta) - LOSS_THRESHOLD,
    'jac' : portfolio_loss_grad,
}

# Starting points: zero (no stress), and pre-stressed directions
rng6 = np.random.default_rng(42)
delta0_list = [
    np.zeros(N_VARS),
    np.array([-1.5, +1.0, +2.0, +0.5, +0.5, +1.0]) * stds,
    np.array([-1.0, +2.0, +2.5, +1.5, +0.5, +0.5]) * stds,
    np.array([-2.0, +0.5, +1.5, +1.0, +1.0, +2.0]) * stds,
    np.array([-0.5, +1.5, +3.0, +2.0, +0.5, +0.5]) * stds,
    np.array([-2.5, +2.0, +1.0, +0.5, +1.0, +1.5]) * stds,
    *(rng6.uniform(-3, 3, (5, N_VARS)) * stds),
]

best = None
for k, d0 in enumerate(delta0_list):
    res = minimize(
        mahal_sq,
        d0,
        method      = 'SLSQP',
        jac         = mahal_sq_grad,
        constraints = constraint,
        options     = dict(maxiter=3000, ftol=1e-12),
    )
    L     = portfolio_loss(res.x)
    D     = np.sqrt(mahal_sq(res.x))
    ok    = L >= LOSS_THRESHOLD - 1.0
    print(f'Run {k+1}: success={res.success}, feasible={ok}, '
          f'D_M={D:.4f},  loss={L:,.0f} EUR m')
    if ok and (best is None or res.fun < best.fun):
        best = res

assert best is not None, 'No feasible solution found'
delta_opt = best.x
D_opt     = np.sqrt(mahal_sq(delta_opt))
L_opt     = portfolio_loss(delta_opt)
x_opt     = mu + delta_opt

print(f'\n✓ Optimal Δ*:  D_M = {D_opt:.4f},  Loss = {L_opt:,.1f} EUR million')

## 7. Optimal Scenario — Results Table

In [ ]:
deviations_sigma = delta_opt / stds

results_df = pd.DataFrame({
    'Variable'          : ALL_VARS,
    'Label'             : [VAR_LABELS[v] for v in ALL_VARS],
    'Hist. mean μ'      : mu,
    'Optimal x* = μ+Δ*' : x_opt,
    'Δ* (raw)'          : delta_opt,
    'Δ* / σ'            : deviations_sigma,
})

print('=' * 90)
print('OPTIMAL MACRO SCENARIO  —  most plausible scenario with portfolio loss ≥ 10,000 EUR m')
print('=' * 90)
print(results_df.to_string(index=False,
                            float_format=lambda v: f'{v:>10.3f}'))

p_chi2  = chi2.cdf(D_opt**2, df=N_VARS)
p_excl  = 1 - p_chi2

print()
print(f'Mahalanobis distance D_M*        = {D_opt:.4f}')
print(f'Squared D_M*  (=  χ² statistic)  = {D_opt**2:.4f}')
print(f'χ²({N_VARS}) tail probability      = {p_excl:.4f}  '
      f'(under MVN, {p_excl*100:.2f}% of macro draws are at least this extreme)')
print()
print(f'Portfolio loss at Δ* (optimum)   = {L_opt:>10,.1f} EUR million')
print(f'Portfolio loss at Δ=0 (baseline) = {loss_base:>10,.1f} EUR million')
print(f'Stress increase                  = {L_opt - loss_base:>10,.1f} EUR million  '
      f'(+{(L_opt/loss_base - 1)*100:.1f}%)')

## 8. Stressed PDs at the Optimal Scenario

In [ ]:
adj_opt     = np.clip(B_total @ delta_opt, -50, 50)
pd_stressed = 1 / (1 + np.exp(-(logit_pd0 + adj_opt)))
pd_stressed = np.clip(pd_stressed, 1e-9, 1 - 1e-9)

df_valid = df_valid.copy()
df_valid['pd_base']     = pd0
df_valid['pd_stressed'] = pd_stressed
df_valid['pd_mult']     = pd_stressed / pd0

tbl_pd = (
    df_valid.groupby('sector')
    .agg(
        n              = ('pd_base', 'count'),
        pd_base_pct    = ('pd_base',     lambda x: f"{x.mean()*100:.2f}%"),
        pd_stress_pct  = ('pd_stressed', lambda x: f"{x.mean()*100:.2f}%"),
        stress_mult    = ('pd_mult',     lambda x: f"{x.mean():.2f}×"),
    )
)
print('Stressed PD summary by sector at optimal Δ*:')
print(tbl_pd.to_string())
print()
print(f'Portfolio-wide base PD    : {pd0.mean()*100:.2f}%')
print(f'Portfolio-wide stressed PD: {pd_stressed.mean()*100:.2f}%')
print(f'Average stress multiplier : {(pd_stressed / pd0).mean():.2f}×')

## 9. Visualisation

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)

# ── A: Scenario deviations (bar chart, σ units) ───────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
colors_a = ['#c0392b' if d < 0 else '#2980b9' for d in deviations_sigma]
bars = ax_a.barh(ALL_VARS, deviations_sigma, color=colors_a, edgecolor='white', height=0.55)
ax_a.axvline(0, color='black', lw=0.8)
for bar, val in zip(bars, deviations_sigma):
    ha = 'left' if val >= 0 else 'right'
    ax_a.text(val + (0.05 if val >= 0 else -0.05),
              bar.get_y() + bar.get_height() / 2,
              f'{val:+.2f}σ', va='center', ha=ha, fontsize=8)
ax_a.set_xlabel('Deviation from μ  (σ units)', fontsize=10)
ax_a.set_title('(A)  Optimal Scenario Deviations', fontweight='bold')
ax_a.grid(axis='x', alpha=0.3)

# ── B: Loss profile along the optimal direction ───────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
direction   = delta_opt / (D_opt + 1e-12)
alphas      = np.linspace(0, D_opt * 1.5, 80)
loss_profile= [portfolio_loss(a * direction) for a in alphas]

ax_b.plot(alphas, loss_profile, lw=2, color='steelblue', label='Loss along optimal direction')
ax_b.axhline(LOSS_THRESHOLD, color='crimson', ls='--', lw=1.5,
             label=f'Threshold {LOSS_THRESHOLD/1000:.0f}k M')
ax_b.axvline(D_opt, color='darkorange', ls=':', lw=1.5,
             label=f'D_M* = {D_opt:.2f}')
ax_b.scatter([D_opt], [L_opt], color='darkorange', zorder=5, s=60)
ax_b.set_xlabel('Mahalanobis distance from μ  (along Δ* direction)', fontsize=10)
ax_b.set_ylabel('Portfolio Loss (EUR m)', fontsize=10)
ax_b.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v/1000:.0f}k'))
ax_b.set_title('(B)  Loss Profile Along Optimal Direction', fontweight='bold')
ax_b.legend(fontsize=8)
ax_b.grid(alpha=0.3)

# ── C: Absolute macro levels: mean vs optimal ─────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
x_pos  = np.arange(N_VARS)
width  = 0.35
ratio  = x_opt / mu
ax_c.bar(x_pos - width/2, np.ones(N_VARS), width, label='Historical mean (= 1)', color='#7fb3d3', edgecolor='white')
ax_c.bar(x_pos + width/2, ratio,            width, label='Optimal x*  (relative)', color='#e59866', edgecolor='white')
ax_c.axhline(1.0, color='black', lw=0.7, ls=':')
ax_c.set_xticks(x_pos)
ax_c.set_xticklabels([v.replace('_', '\n') for v in ALL_VARS], fontsize=8)
ax_c.set_ylabel('Value relative to historical mean', fontsize=9)
ax_c.set_title('(C)  Macro Levels: Mean vs. Optimal Scenario', fontweight='bold')
ax_c.legend(fontsize=8)
ax_c.grid(axis='y', alpha=0.3)

# ── D: PD distribution base vs stressed ───────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])
pd_max_pct = min(max(pd_stressed.max(), pd0.max()) * 100 * 1.1, 100)
bins = np.linspace(0, pd_max_pct, 55)
ax_d.hist(pd0       * 100, bins=bins, alpha=0.55, label='Base PD',     color='#7fb3d3', edgecolor='white')
ax_d.hist(pd_stressed * 100, bins=bins, alpha=0.55, label='Stressed PD', color='#e59866', edgecolor='white')
ax_d.axvline(pd0.mean()        * 100, color='steelblue', ls='--', lw=1.5,
             label=f'Mean base  {pd0.mean()*100:.2f}%')
ax_d.axvline(pd_stressed.mean() * 100, color='darkorange', ls='--', lw=1.5,
             label=f'Mean stressed  {pd_stressed.mean()*100:.2f}%')
ax_d.set_xlabel('PD (%)', fontsize=10)
ax_d.set_ylabel('Number of exposures', fontsize=10)
ax_d.set_title('(D)  PD Distribution: Base vs. Stressed', fontweight='bold')
ax_d.legend(fontsize=8)
ax_d.grid(alpha=0.3)

fig.suptitle(
    f'Constrained Macro Optimisation  —  Most Plausible Scenario with Loss ≥ {LOSS_THRESHOLD:,.0f} EUR million\n'
    f'Mahalanobis distance  D_M* = {D_opt:.3f}    |    '
    f'χ²({N_VARS}) tail prob. = {p_excl*100:.2f}%    |    '
    f'Loss at x* = {L_opt:,.0f} EUR million',
    fontsize=11, fontweight='bold',
)
plt.savefig('constrained_opt_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → constrained_opt_results.png')

## 10. Pareto Frontier — Plausibility vs. Severity

In [ ]:
thresholds  = np.arange(3_000, 14_001, 500, dtype=float)
front_D     = []
front_L     = []
front_ok    = []
delta_warm  = np.zeros(N_VARS)

for thr in thresholds:
    con = {'type': 'ineq', 'fun': lambda d, t=thr: portfolio_loss(d) - t,
           'jac': portfolio_loss_grad}
    res = minimize(mahal_sq, delta_warm, method='SLSQP',
                   jac=mahal_sq_grad, constraints=con,
                   options=dict(maxiter=3000, ftol=1e-12))
    L   = portfolio_loss(res.x)
    D   = np.sqrt(mahal_sq(res.x))
    ok  = L >= thr - 5.0
    front_D.append(D)
    front_L.append(L)
    front_ok.append(ok)
    if ok:
        delta_warm = res.x

ok_idx   = np.where(front_ok)[0]
thr_ok   = thresholds[ok_idx] / 1000
D_ok     = np.array(front_D)[ok_idx]
p_ok     = (1 - chi2.cdf(D_ok**2, df=N_VARS)) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(thr_ok, D_ok, 'o-', color='steelblue', lw=2, ms=5)
ax1.axvline(LOSS_THRESHOLD / 1000, color='crimson', ls='--', lw=1.5,
            label=f'Target {LOSS_THRESHOLD/1000:.0f}k M')
ax1.axhline(D_opt, color='darkorange', ls=':', lw=1.5, label=f'D_M* = {D_opt:.2f}')
ax1.set_xlabel('Loss Threshold (EUR billion)', fontsize=11)
ax1.set_ylabel('Mahalanobis Distance D_M', fontsize=11)
ax1.set_title('Plausibility of Worst-Case Scenario\nvs. Loss Severity', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.plot(thr_ok, p_ok, 's-', color='seagreen', lw=2, ms=5)
ax2.axvline(LOSS_THRESHOLD / 1000, color='crimson', ls='--', lw=1.5,
            label=f'Target {LOSS_THRESHOLD/1000:.0f}k M')
ax2.set_xlabel('Loss Threshold (EUR billion)', fontsize=11)
ax2.set_ylabel('Scenario tail probability  [χ² tail, %]', fontsize=11)
ax2.set_title('Scenario Probability\nvs. Loss Severity', fontweight='bold')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

fig.suptitle('Pareto Frontier: Loss Threshold vs. Scenario Plausibility',
             fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('constrained_opt_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → constrained_opt_frontier.png')

## Summary

| | |
|---|---|
| **Loss threshold** | 10,000 EUR million |
| **Baseline loss** (no macro stress, q = 99.9%) | 2,410 EUR million |
| **Optimal Mahalanobis distance** D_M* | see cell above |
| **Scenario probability** (χ²(5) tail) | see cell above |
| **Gordy quantile** | 99.9% |
| **LGD** | 40% |

### Key design choices

1. **Long-run sensitivities**: Summing current + 4 lagged beta coefficients gives the  
   steady-state PD response to a permanent macro shock — more meaningful for scenario analysis  
   than using only the current-period coefficient in a multicollinear lag regression.

2. **Deviation formulation**: The optimization is over $\boldsymbol{\Delta} = \mathbf{x} - \boldsymbol{\mu}$,  
   so at $\boldsymbol{\Delta} = 0$ the portfolio uses its base PDs and the loss is the Gordy VaR  
   at the historical macro average — a natural anchor point.

3. **Mahalanobis plausibility**: Under a multivariate-normal approximation of the macro distribution,  
   the χ²(5) tail probability gives the fraction of historical macro observations that would be  
   at least as extreme as the optimal stress scenario.

4. **Pareto frontier** (section 10): Sweeping the loss threshold from 3,000 to 14,000 EUR million  
   traces the full trade-off between scenario plausibility and loss severity.

## 11. Hurlin et al. Figures 1 & 2 — Loss Frontier Geometry

The two panels below replicate the geometric structure of **Figures 1 and 2** in Hurlin et al. (2026),
translated to our absolute-loss framework.

We project into the **two scenario dimensions with the largest |Δ/σ|** at the 6-D design point,
holding all other variables at their historical baseline (Δ = 0). This gives a 2-D slice of the
full optimisation problem in which the geometry is most visible.

| Hurlin et al. element | Our analog |
|---|---|
| Breakdown frontier ∂S_red = {s : R(s) = Rω} | Loss frontier {Δ : L(Δ) = thr} |
| Breach region S_red (shaded) | {Δ : L(Δ) ≥ thr} |
| Plausibility contours d²_Σ(s) = const | Mahalanobis ellipses d²_Σ(Δ) = const centred at Δ=0 |
| Design point sω (tangency point) | Design point δ* (minimum D_M on the frontier) |
| Local ball B_ρ(sω) [Fig. 1] | Mahalanobis ball of radius² ρ centred at δ* |
| Near-optimal set N_ε [Fig. 2] | {Δ : L ≥ thr AND d²_Σ ≤ d²_Σ(δ*) + ε} |

> To use the Hurlin (x, GPR) plane directly, set `idx_b = ALL_VARS.index('GPR_Global')` below.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ── Variable selection: two most impactful dimensions at the 6-D design point ─
# To use the Hurlin (x, GPR) axes, replace the idx_b line with:
#   idx_b = ALL_VARS.index('GPR_Global')
top2  = np.argsort(np.abs(deviations_sigma))[-2:][::-1]
idx_a = int(top2[0])   # most impactful
idx_b = int(top2[1])   # second most impactful
lbl_a = VAR_LABELS[ALL_VARS[idx_a]]
lbl_b = VAR_LABELS[ALL_VARS[idx_b]]
print(f'2-D slice:  x = {lbl_a}  |Δ/σ| = {abs(deviations_sigma[idx_a]):.2f}')
print(f'            y = {lbl_b}  |Δ/σ| = {abs(deviations_sigma[idx_b]):.2f}')

# ── 2-D grid (all other scenario variables held at Δ = 0) ────────────────────
N_G    = 160                           # higher resolution for near-optimal region
rng_a  = 4.5 * stds[idx_a];  rng_b = 4.5 * stds[idx_b]
ga     = np.linspace(-rng_a, rng_a, N_G)
gb     = np.linspace(-rng_b, rng_b, N_G)
GA, GB = np.meshgrid(ga, gb)                  # (N_G, N_G)

# Vectorised portfolio loss on the 2-D slice ──────────────────────────────────
b_a = B_total[:, idx_a];  b_b = B_total[:, idx_b]
da_f = GA.ravel()[:, None];  db_f = GB.ravel()[:, None]     # (N²,1)
adj_f = np.clip(logit_pd0[None,:] + da_f * b_a[None,:] + db_f * b_b[None,:], -50, 50)
pd_f  = np.clip(1 / (1 + np.exp(-adj_f)), 1e-9, 1-1e-9)
cp_f  = norm.cdf((norm.ppf(pd_f) + sqrt_rho[None,:] * inv_q) / sqrt_1mrho[None,:])
L_2D  = (ead[None,:] * LGD * cp_f).sum(1).reshape(N_G, N_G)

# Vectorised Mahalanobis d²(Δ) from baseline Δ=0 ─────────────────────────────
dfl           = np.zeros((N_G * N_G, N_VARS))
dfl[:, idx_a] = GA.ravel();  dfl[:, idx_b] = GB.ravel()
D2_2D         = ((dfl @ Sigma_inv) * dfl).sum(1).reshape(N_G, N_G)

# ── 2-D design point ─────────────────────────────────────────────────────────
thr_2d = min(LOSS_THRESHOLD, float(L_2D.max()) * 0.95)
if thr_2d < LOSS_THRESHOLD - 1:
    print(f'\n  ⚠ 2-D max loss {L_2D.max():,.0f} < {LOSS_THRESHOLD:,.0f}; '
          f'using thr_2d = {thr_2d:,.0f} EUR m')
else:
    thr_2d = LOSS_THRESHOLD

def _l2(ab):
    d = np.zeros(N_VARS); d[idx_a]=ab[0]; d[idx_b]=ab[1]; return portfolio_loss(d)
def _d2(ab):
    d = np.zeros(N_VARS); d[idx_a]=ab[0]; d[idx_b]=ab[1]; return mahal_sq(d)

con2  = {'type': 'ineq', 'fun': lambda ab: _l2(ab) - thr_2d}
best2 = None
for ini in [np.zeros(2),
            delta_opt[[idx_a, idx_b]],
            np.array([-1.5*stds[idx_a],  2.0*stds[idx_b]]),
            np.array([-2.5*stds[idx_a],  0.5*stds[idx_b]])]:
    r = minimize(_d2, ini, method='SLSQP', constraints=con2,
                 options=dict(maxiter=5000, ftol=1e-14))
    if _l2(r.x) >= thr_2d - 1 and (best2 is None or r.fun < best2.fun):
        best2 = r

ab_star  = best2.x
d2_star  = _d2(ab_star);  dm_star = np.sqrt(d2_star)
l_star   = _l2(ab_star)
print(f'\n  2-D design point: {lbl_a} = {ab_star[0]/stds[idx_a]:+.2f}σ, '
      f'{lbl_b} = {ab_star[1]/stds[idx_b]:+.2f}σ')
print(f'  D_M = {dm_star:.3f},  loss = {l_star:,.0f} EUR m')

# Mahalanobis centred at design point (for local neighbourhood ball) ──────────
dfl2           = np.zeros((N_G * N_G, N_VARS))
dfl2[:, idx_a] = GA.ravel() - ab_star[0];  dfl2[:, idx_b] = GB.ravel() - ab_star[1]
D2_dpt         = ((dfl2 @ Sigma_inv) * dfl2).sum(1).reshape(N_G, N_G)

rho_ball = d2_star * 0.20        # local ball radius²  ρ
epsilon  = d2_star * 1.50        # near-optimal tolerance ε (≈ 2.5× d²*)
near_opt = (L_2D >= thr_2d) & (D2_2D <= d2_star + epsilon)

# Grid in σ-units for axis labels
gx = GA / stds[idx_a];  gy = GB / stds[idx_b]
axs = ab_star[0] / stds[idx_a];  ays = ab_star[1] / stds[idx_b]

# d² contour levels: a few inner curves + the tangent level
inner_levs = np.array([0.20, 0.45, 0.70, 0.90]) * d2_star

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 ANALOG — loss frontier, plausibility contours, local neighbourhood
# ═══════════════════════════════════════════════════════════════════════════════
fig1, ax1 = plt.subplots(figsize=(7.5, 6.5))

ax1.contourf(gx, gy, L_2D, levels=[thr_2d, L_2D.max()+1e4],
             colors=['#fdb8b8'], alpha=0.55)
ax1.contour(gx, gy, L_2D, levels=[thr_2d],
            colors=['crimson'], linewidths=2.5)
ax1.contour(gx, gy, D2_2D, levels=inner_levs,
            colors='steelblue', linewidths=0.9, linestyles='--', alpha=0.55)
ax1.contour(gx, gy, D2_2D, levels=[d2_star],
            colors='steelblue', linewidths=2.2, linestyles='--')
ax1.contour(gx, gy, D2_dpt, levels=[rho_ball],
            colors='darkorange', linewidths=1.8, linestyles=':')
ax1.plot(axs, ays, 'k*', ms=15, zorder=6)
ax1.plot(0,   0,   'ko', ms=7,  zorder=5)
ax1.axhline(0, color='k', lw=0.5, ls=':')
ax1.axvline(0, color='k', lw=0.5, ls=':')

ax1.legend(handles=[
    Patch(fc='#fdb8b8', ec='crimson', alpha=0.75,
          label='Breach region  $\\mathcal{S}_{\\mathrm{red}}$'),
    Line2D([0],[0], color='crimson', lw=2.5,
           label=f'Loss frontier  $L = {thr_2d/1000:.0f}$k EUR M'),
    Line2D([0],[0], color='steelblue', lw=1.2, ls='--', alpha=0.7,
           label='Plausibility iso-curves  $d^2_\\Sigma(\\Delta)$'),
    Line2D([0],[0], color='steelblue', lw=2.2, ls='--',
           label=f'Tangent curve  $d^2 = {d2_star:.1f}$'),
    Line2D([0],[0], color='darkorange', lw=1.8, ls=':',
           label=f'Local ball  $B_\\rho$  ($\\rho^2 = {rho_ball:.1f}$)'),
    Line2D([0],[0], marker='*', color='k', ms=11, ls='',
           label=f'Design point  $\\delta^*$  ($D_M = {dm_star:.2f}$)'),
    Line2D([0],[0], marker='o', color='k', ms=7, ls='',
           label='Baseline  ($\\Delta = \\mathbf{0}$)'),
], fontsize=8.5, loc='best')

ax1.set_xlabel(f'$\\Delta$ {lbl_a}  (σ-units)', fontsize=11)
ax1.set_ylabel(f'$\\Delta$ {lbl_b}  (σ-units)', fontsize=11)
ax1.set_title(
    'Loss frontier in the scenario plane\n'
    f'(analog of Hurlin et al. Fig. 1  —  threshold = {thr_2d/1000:.0f}k EUR M)',
    fontsize=11, fontweight='bold')
ax1.set_xlim(gx.min(), gx.max());  ax1.set_ylim(gy.min(), gy.max())
ax1.grid(alpha=0.2)
fig1.tight_layout()
plt.savefig('hurlin_fig1_analog.png', dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 ANALOG — adds near-optimal set N_ε
# ═══════════════════════════════════════════════════════════════════════════════
fig2, ax2 = plt.subplots(figsize=(7.5, 6.5))

ax2.contourf(gx, gy, L_2D, levels=[thr_2d, L_2D.max()+1e4],
             colors=['#fdb8b8'], alpha=0.40)
ax2.contourf(gx, gy, near_opt.astype(float), levels=[0.5, 1.5],
             colors=['#c0392b'], alpha=0.35, hatches=['////'])
ax2.contour(gx, gy, L_2D, levels=[thr_2d],
            colors=['crimson'], linewidths=2.5)
ax2.contour(gx, gy, D2_2D, levels=inner_levs,
            colors='steelblue', linewidths=0.9, linestyles='--', alpha=0.55)
ax2.contour(gx, gy, D2_2D, levels=[d2_star],
            colors='steelblue', linewidths=2.2, linestyles='--')
ax2.contour(gx, gy, D2_2D, levels=[d2_star + epsilon],
            colors='steelblue', linewidths=2.0, linestyles='-.')
ax2.plot(axs, ays, 'k*', ms=15, zorder=6)
ax2.plot(0,   0,   'ko', ms=7,  zorder=5)
ax2.axhline(0, color='k', lw=0.5, ls=':')
ax2.axvline(0, color='k', lw=0.5, ls=':')

ax2.legend(handles=[
    Patch(fc='#fdb8b8', ec='crimson', alpha=0.60,
          label='Breach region  $\\mathcal{S}_{\\mathrm{red}}$'),
    Patch(fc='#c0392b', ec='#c0392b', alpha=0.45, hatch='////',
          label=f'Near-optimal set  $\\mathcal{{N}}_\\varepsilon$  ($\\varepsilon = {epsilon:.1f}$)'),
    Line2D([0],[0], color='crimson', lw=2.5,
           label=f'Loss frontier  $L = {thr_2d/1000:.0f}$k EUR M'),
    Line2D([0],[0], color='steelblue', lw=1.2, ls='--', alpha=0.7,
           label='Plausibility iso-curves  $d^2_\\Sigma(\\Delta)$'),
    Line2D([0],[0], color='steelblue', lw=2.2, ls='--',
           label=f'Tangent curve  $d^2 = {d2_star:.1f}$'),
    Line2D([0],[0], color='steelblue', lw=2.0, ls='-.',
           label=f'Outer contour  $d^2 = {d2_star+epsilon:.1f}$'),
    Line2D([0],[0], marker='*', color='k', ms=11, ls='',
           label=f'Design point  $\\delta^*$  ($D_M = {dm_star:.2f}$)'),
], fontsize=8.5, loc='best')

ax2.set_xlabel(f'$\\Delta$ {lbl_a}  (σ-units)', fontsize=11)
ax2.set_ylabel(f'$\\Delta$ {lbl_b}  (σ-units)', fontsize=11)
ax2.set_title(
    'Near-optimal scenario set  $\\mathcal{N}_\\varepsilon$\n'
    f'(analog of Hurlin et al. Fig. 2  —  $\\varepsilon = {epsilon:.1f}$)',
    fontsize=11, fontweight='bold')
ax2.set_xlim(gx.min(), gx.max());  ax2.set_ylim(gy.min(), gy.max())
ax2.grid(alpha=0.2)
fig2.tight_layout()
plt.savefig('hurlin_fig2_analog.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figures saved  →  hurlin_fig1_analog.png,  hurlin_fig2_analog.png')